# PT3: CUDA and HIP Metaprogramming and Kernel Depth

Given what we've learned about padding dimensions, the GPU block sizes, and the GPU grid dimensions, as well as how each path is mapped using `_compile_for_device(self, device)`. We're now able to go in depth inside `pooling_kernel.py`. 

In the last notebook, I had discussed about the four helper methods below:

In [ ]:
from string import Template
import aether.config as config

# Our template strings — 
_POOL_FORWARD_TEMPLATE = Template(r''' ''')
_MAX_BACKWARD_NONOVERLAP_TEMPLATE = Template(r''' ''')
_AVG_BACKWARD_NONOVERLAP_TEMPLATE = Template(r''' ''') 
# Our op dicts — C++ snippets that plug into _POOL_FORWARD_TEMPLATE
_MAX_OP = {}
_MAX_OP_INFERENCE = {}
_AVG_OP = {}

def _get_compiled_forward_kernel(op, variant):
    pass 
def _get_compiled_max_backward_kernel(variant):
    pass
def _get_compiled_avg_backward_kernel(variant):
    pass
# Helper Functions for MaxPool2D
def is_gpu_max_pool2d_available() -> bool:
    """Checks if CuPy hardware support and kernels are loaded."""
    return config.HAS_CUPY

def get_max_pool2d_forward_kernel(variant: str, training: bool = True):
    op = _MAX_OP if training else _MAX_OP_INFERENCE
    return _get_compiled_forward_kernel(op, variant)

def get_max_pool2d_backward_kernel(variant: str):
    return _get_compiled_max_backward_kernel(variant)
# Helper Functions for AvgPool2d
def is_gpu_avg_pool2d_available():
    return config.HAS_CUPY

def get_avg_pool2d_forward_kernel(variant: str):
    return _get_compiled_forward_kernel(_AVG_OP, variant)

def get_avg_pool2d_backward_kernel(variant: str):
    return _get_compiled_avg_backward_kernel(variant)

The for `get` methods at the bottom are self explanatory, while the two `is_gpu` methods are simple checks for `_compile_for_device`. I likely will refactor the latter part. 

The three interesting methods are the `get_compiled_` methods at the top. We'll explore why the forward pass is used for both the max and average pooling forward passes, but the backward passes are seperated into their own methods.

In [ ]:
_pool_kernel_cache = {}
def _get_compiled_forward_kernel(op_dict: dict, variant: str):
    cache_key = (op_dict["name"], variant)
    if cache_key in _pool_kernel_cache:
        return _pool_kernel_cache[cache_key]

    kernel_name = f"pool2d_forward_{op_dict['name']}_{variant}_kernel"
    source = _POOL_FORWARD_TEMPLATE.substitute(
        hip_include="#include <hip/hip_runtime.h>\n" if variant == "hip" else "",
        kernel_name=kernel_name,
        aux_out_decl=op_dict["aux_out_decl"],
        reduce_init=op_dict["reduce_init"],
        reduce_body=op_dict["reduce_body"],
        writeback=op_dict["writeback"],
    )

    kernel = config.build_kernel(
        lambda: config.cp.RawKernel(source, kernel_name),
        name=f"pool2d_forward_{op_dict['name']}_{variant}",
    )
    _pool_kernel_cache[cache_key] = kernel
    return kernel
def _get_compiled_max_backward_kernel(variant: str):
    pass
def _get_compiled_avg_backward_kernel(variant: str):
    pass


### So Why Even Use Templates? Why Memoize the Kernels As Well? 

`cupy.RawKernel`s allow the programmer to write a kernel from raw CUDA source. However, because the framework needs to support HIP backends, we need a way to write our `RawKernel`s to support both backend architectures. This is the rationale behind having `variant: str` be present across all the `_get_` functions. This lets the backend automatically include the line inside the `source` variable found in all the files. 

```python
cache_key = (op_dict["name"], variant)
if cache_key in _pool_kernel_cache:
    return _pool_kernel_cache[cache_key]
```
This is the part that we memoize, we use an $O(1)$ lookup to check for our kernel inside the `_pool_kernel_cache` dictionary, and return our kernel (inside `pooling.py`, this is why we can straight call `kernel` like a normal function). Otherwise, we'll have to build the actual kernel, this involves mesing with the parameters of our strings, as well as having a custom name.

We encapsulate the logic into small dictionaries `_MAX_OP`, `_MAX_OP_INFERENCE`, and `_AVG_OP`. 
```python
    kernel_name = f"pool2d_forward_{op_dict['name']}_{variant}_kernel"
    source = _POOL_FORWARD_TEMPLATE.substitute(
        hip_include="#include <hip/hip_runtime.h>\n" if variant == "hip" else "",
        kernel_name=kernel_name,
        aux_out_decl=op_dict["aux_out_decl"],
        reduce_init=op_dict["reduce_init"],
        reduce_body=op_dict["reduce_body"],
        writeback=op_dict["writeback"],
    )
```
With the Python `Template` class, we can inject variables using `$`, and create branches using the variable. For exmaple: 
```python
_POOL_FORWARD_TEMPLATE = Template(r'''
$hip_include
extern "C" __global__
void $kernel_name(
// rest of code ... 
```
* `hip_include`: We have the variable `hip_include` which is needed for runing the kernel via the **HIPRTC** (HIP runtime compiler). If we're on CUDA, we'll omit this as CuPy defaults to **NVRTC** anyways.
* `kernel_name`: We can substitute the name with an actual name, examples of this would be:
    1. `pool2d_forward__MAX_OP_hip`
    2. `pool2d_forward__MAX_OP_INFERENCE_cuda`

The entire template string and max op dictionaries are shown below, to continue our notes: 

In [ ]:
from string import Template
import aether.config as config

_POOL_FORWARD_TEMPLATE = Template(r'''
$hip_include
extern "C" __global__
void $kernel_name(
    const float* __restrict__ x,
    float* __restrict__ out,
    $aux_out_decl
    const int S, const int H_pad, const int W_pad, const int C,
    const int fH, const int fW, const int sH, const int sW,
    const int H_out, const int W_out,
    const unsigned int magic_scale, const int magic_shift  
) {
    int c = blockIdx.x * blockDim.x + threadIdx.x;
    int w_out = blockIdx.y * blockDim.y + threadIdx.y;
    if (c >= C || w_out >= W_out) return;

    int h_s = blockIdx.z * blockDim.z + threadIdx.z;
    unsigned long long prod = (unsigned long long)h_s * magic_scale;
    int s = (int)(prod >> (32 + magic_shift));
    if (s >= S) return;

    int h_out = h_s - (s * H_out);

    int h_start = h_out * sH;
    int w_start = w_out * sW;
    int batch_offset = s * H_pad * W_pad * C;

    $reduce_init

    for (int fh = 0; fh < fH; ++fh) {
        int h_in = h_start + fh;
        int row_offset = batch_offset + (h_in * W_pad) * C;

        for (int fw = 0; fw < fW; ++fw) {
            int w_in = w_start + fw;
            int in_idx = row_offset + w_in * C + c;

            $reduce_body
        }
    }

    int out_idx = ((s * H_out + h_out) * W_out + w_out) * C + c;
    $writeback
}
''')

_MAX_OP = {
    "name": "max",
    "aux_out_decl": "int* __restrict__ max_indices,",
    "reduce_init": (
        "int initial_idx = batch_offset + (h_start * W_pad + w_start) * C + c;\n"
        "    float max_val = x[initial_idx];\n"
        "    int best_idx = initial_idx;"
    ),
    "reduce_body": (
        "float val = x[in_idx];\n"
        "            if (val > max_val) { max_val = val; best_idx = in_idx; }"
    ),
    "writeback": "out[out_idx] = max_val;\n    max_indices[out_idx] = best_idx;",
}

_MAX_OP_INFERENCE = {
    "name": "max_inference",
    "aux_out_decl": "",
    "reduce_init": (
        "int initial_idx = batch_offset + (h_start * W_pad + w_start) * C + c;\n"
        "    float max_val = x[initial_idx];"
    ),
    "reduce_body": (
        "float val = x[in_idx];\n"
        "            if (val > max_val) { max_val = val; }"
    ),
    "writeback": "out[out_idx] = max_val;",
}

_AVG_OP = {
    "name": "avg",
    "aux_out_decl": "",
    "reduce_init": "float sum_val = 0.0f;",
    "reduce_body": "sum_val += x[in_idx];",
    "writeback": "out[out_idx] = sum_val / (float)(fH * fW);",
}

* `aux_out_decl`: These injects extra parameter pointers, for `_MAX_OP`, we inject the pointer `"int* __restrict__ max_indices`
    1. `__restrict__`: We use this type qualifier to assert that the memory space pointed to by `max_indices` does not overlap with any other pointer passed into the function. At a high level if we can include `__restrict__`, then we get free performance from the compiler. 
* `reduce_init`: Prepares accumulators before entering the spatial loop. This could include initalizing `sum_val = 0.0f` vs tracking `max_val = x[initial_idx]`. 
    2. Inside the template body, its by itself `reduce_init` right before the for loop that tracks over the `fH` and `fW` elements. 
* `reduce_body`: Executes the reduction per window element. In max pooling, we'd update the running maximum and argmax index, but in average pooling we'd run `float val = x[in_idx];` instead. 
* `writeback`: This final string subsitution defines the output handling. This is only important for the inference part of the max pooling operation since we don't want to save `self.max_indices`. 

## A Concrete Example of the Template Substitution in Action

When `_get_compiled_forward_kernel(_MAX_OP, variant="cuda")` is executed, we'll create a template string that substitutes the placeholders with code blocks from `_MAX_OP`. I'll add comments wherever the changes are made.

In [ ]:
_POOL_FORWARD_TEMPLATE = Template(r'''
# The $hip_include gets replaced with "" because we are using variant cuda
extern "C" __global__
void pool2d_forward__MAX_OP_cuda(
    const float* __restrict__ x,
    float* __restrict__ out,
    int* __restrict__ max_indices, #$aux_out_decl gets replaced with the max_indices
    const int S, const int H_pad, const int W_pad, const int C,
    const int fH, const int fW, const int sH, const int sW,
    const int H_out, const int W_out,
    const unsigned int magic_scale, const int magic_shift  
) {
    int c = blockIdx.x * blockDim.x + threadIdx.x;
    int w_out = blockIdx.y * blockDim.y + threadIdx.y;
    if (c >= C || w_out >= W_out) return;

    int h_s = blockIdx.z * blockDim.z + threadIdx.z;
    unsigned long long prod = (unsigned long long)h_s * magic_scale;
    int s = (int)(prod >> (32 + magic_shift));
    if (s >= S) return;

    int h_out = h_s - (s * H_out);

    int h_start = h_out * sH;
    int w_start = w_out * sW;
    int batch_offset = s * H_pad * W_pad * C;

    # $reduce_init. These three lines find our initial index for the thread
    # the max value (we start at the top left)
    # and initialize the best index
    int initial_idx = batch_offset + (h_start * W_in + w_start) * C + c;
    float max_val = x[initial_idx];
    int best_idx = initial_idx;

    # We then run through the fHxfW window to find the best index
    for (int fh = 0; fh < fH; ++fh) {
        int h_in = h_start + fh;
        int row_offset = batch_offset + (h_in * W_pad) * C;

        for (int fw = 0; fw < fW; ++fw) {
            int w_in = w_start + fw;
            int in_idx = row_offset + w_in * C + c;

            # $reduce_body
            float val = x[in_idx];
            if (val > max_val) { max_val = val; best_idx = in_idx;
        }
    }

    int out_idx = ((s * H_out + h_out) * W_out + w_out) * C + c;
    $writeback
}
''')

### How does the Algorithm Inside the Kernel Work? 

We have three main bodies to account for which are the three templates
* `_POOL_FORWARD_TEMPLATE`: Needed for the forward pass for both max and average pooling,  along with specfic `OP` dicts that are meant for max pooling inference/training.

* `_MAX_BACKWARD_DIRECT_TEMPLATE`: A dedicated backward pass kernel that specifically routes the upstream `dvalues` to the downstream `dinputs` with the assumption that `fH == sH` and `fW == sW`. With this scenario, we don't have overlapping writes with our dinputs, meaning we can avoid the use of atomic operations with `cp.atomicAdd`. 

* `_AVG_BACKWARD_NONOVERLAP_TEMPLATE`: A dedicated backward pass kernel with the same `fH == sH` and `fW == sW` assumption. In this scenario, we can safely divide the window area 
 ($\frac{d_{values}}{f_H \times f_W}$) across the windows input locations. 

All three are quite important, we'll be thorough in explaining the decisions inside `_POOL_FORWARD_TEMPLATE`, and quickly go over `_MAX_BACKWARD_DIRECT_TEMPLATE` and `_AVG_BACKWARD_NONOVERLAP_TEMPLATE`. 

# _POOL_FORWARED_TEMPLATE Walkthrough

There are four main stages to the kernel: Thread Grid Mapping, Receptive Field Setup, Spatial Window Reduction, and Global Writeback.

At the top, we'll create our input variables for the kernel. Major note, the tensor dimensions will be captialized `H`, `W`, `C`, `S`, `H_out`, `W_out`. We'll use these, along with the thread coordinates `h`, `w`, `c`, `s`, `h_out`, `w_out` which represents the specific element location being assigned to a thread. This has to be computed at runtime but its cheap to compute. 

```python
_POOL_FORWARD_TEMPLATE = Template(r'''
$hip_include
extern "C" __global__
void $kernel_name(
    const float* __restrict__ x,
    float* __restrict__ out,
    $aux_out_decl
    const int S, const int H_pad, const int W_pad, const int C,
    const int fH, const int fW, const int sH, const int sW,
    const int H_out, const int W_out,
    const unsigned int magic_scale, const int magic_shift  
```

* `__global__`: When we mark a function with `__global__`, we delegate that the function will be ran entirely on the GPU (its a kernel) 
We manually defined what each of our inputs were, which are all passed as a dictionary on the python side. 
`const unsigned int magic_scale, const int magic_shift`: These are two values needed to unpack the `grid_z` dimension so that we can flesh out our batch and output height thread . 

* `extern "C" __global`: C++ compilers perform **name mangling** (which alter function names to support function overloading). When we add `extern "C"`, we instruct our GPU compiler to use C-style linkage. This ensures CuPy can find a specific kernel such as `pool2d_forward_max_cuda_kernel` by its exact string name in memory.


## Stage 1: Global Thread Index Reconstruction & Boundary Guards

In PT2, we explained how we set up the 3D grid dimensions as well as the block dimensions for our kernel. The point of that was for our kernel to be mapped to thousands of parallel threads across a 3D layout. As a quick reminder, the block dimensions decide our maximum amount of threads per grid. Our grid dimensions are where we stack our blocks across the 4D tensor for the dataset. 

Each GPU thread is assigned to compute exactly **one output element** at 1D coordinate 
$$\text{out\_idx} = ((s \times H_{\text{out}} + h_{\text{out}}) \times W_{\text{out}} + w_{\text{out}}) \times C + c$$
The GPU driver dispatches threads across a 3D grid. The kernel first decodes its global execution coordinates from `blockIdx`, `blockDim`, and `threadIdx` (built in variables that each thread has access to):

* `threadIdx`: Its position *within* its assigned block $(x, y, z)$. 
* `blockIdx`: Its position *within* the dataset itself, which is based off of its position within the grid $(x, y, z)$.
* `blockDim`: This gives us the size/dimensions of each block. 

Inside the code, we need to unpack these such that the thread knows exactly where it needs to perform its operation over a $fH \times fW$ field.
```python
int c = blockIdx.x * blockDim.x + threadIdx.x;      // Channel dimension (Global X coordinate)
int w_out = blockIdx.y * blockDim.y + threadIdx.y;  // Output spatial width (Global Y coordinate)
int h_s = blockIdx.z * blockDim.z + threadIdx.z;    // Fused (Batch * Output Height) (Global Z coordinate)

int h_out = h_s % H_out;
int s = h_s / H_out;
```
Unfusing Dimensions: Because CUDA/HIP limits grid dimensions (4D tensor compressed into 3D grid), batch size $S$ and output height $H_{out}$ are fused into `h_s`. 
$$h_s = s \times H_{\text{out}} + h_{\text{out}}$$

To extract $s$ and $h_{out}$ back out inside the thread, the thread unrolls them via integer division and modulus (in other words we reverse the fusion):
* `h_out = h_s % H_out` (row index within the feature map)
* `s = h_s / H_out` (sample index within the batch)
    * Note: Integer division inside the GPU is slow because GPUs lack dedicated hardware integer dividers. This means we take maybe **10x** or more **clock cycles** to perform these two operations over a standard `FMUL` or `FADD`. The good news is, we only compute this once per thread. The bad news is, we still compute this once once per thread. 

Since we're still going to write a `cupy.RawKernel` for a `Conv` layer, it makes sense to minimize this overhead, as we still need to unpack a $4D$ tensor from a $3D$ grid. GPUs still find the result using **magic-nubmer division**. Because the general shape of the input never changes during a specific pooling layer, we'll memoize the output or cache the result avoiding any CPU overhead per forward/backward call.  

Instead of running `s = h_s / H_out`, the CPU precomputes two parameters based on $H_{out}$:
1. `magic_scale`: A 32-bit fixed-point multiplier equivalent to  $\lceil \frac{2^{32 + \text{shift}}}{H_{out}} \rceil$.
2. `magic_shift`: A bit-shift offset (we learned about bit shift offsets inside the custom philox kernel)
Now, the kernel spends around 4-5 clock cycles unpacking this instead of the 30+ it used to for the integer division. 

```c++
int h_s = blockIdx.z * blockDim.z + threadIdx.z;

// High-speed fixed-point division: calculates s = h_s / H_out
unsigned long long prod = (unsigned long long)h_s * magic_scale;
int s = (int)(prod >> (32 + magic_shift));
if (s >= S) return;

// Fast recovery of h_out without modulo: h_out = h_s - (s * H_out)
int h_out = h_s - (s * H_out);
```

The actual derivation of this doesn't really matter, as either way for any other template or when we write the custom kernel for `Conv`, it will have this exact same logic unpacked. 

## Stage 2: Receptive Field Setup

$$\text{out\_idx} = \big((s \times H_{\text{out}} + h_{\text{out}}) \times W_{\text{out}} + w_{\text{out}}\big) \times C + c$$

$\text{out\_idx}$ tells us where we are inside the tensor. This gives us the linear memory offset in SHWC layout using a few nested operations. The equation above is what the GPU reads to maximize GPU instruction counts (we only require 3 `MUL` and 3 `ADD` over an expanded 6 `MUL` and 3 `ADD`). Basically, less instructions = faster. 

We'll distribute $W_{out}$ followed by distributing $C$ afterwards.
$$\text{out\_idx} = (s \cdot H_{\text{out}} \cdot W_{\text{out}} + h_{\text{out}} \cdot W_{\text{out}} + w_{\text{out}}) \cdot C + c$$

$$\text{out\_idx} = s \cdot (H_{\text{out}} \cdot W_{\text{out}} \cdot C) + h_{\text{out}} \cdot (W_{\text{out}} \cdot C) + w_{\text{out}} \cdot C + c$$

This is the version we can better explain. This will break down the physical memory into array strides. 

When we created that 4D tensor of shape ($S, H, W, C$), we were fitting a 4D structure into a 1D straight line. This equation is our way of deciding **how many elements we must jump over** in that 1D line to reach the exact item at coordinate ($s, h, w, c$). 

1. Step into target Channel $(c)$:
    * Unit size: 1 float element
    * Individual channels live right next to each other in memory (because of the SHWC layout), moving from channel $n$ to channel $n+1$ means jumping **1 element forward**.
    * **Offset cost:** $c\times1$

2. Skip previous pixels ($w$):
    * Unit size: $C$ float elements
    * Each pixel is a bundle containing $C$ channels. Moving to the next pixel on the right ($w + 1$) means we'll have to jump over all $C$ channels of the current pixel.
    * **Offset cost:** $w \times C$
3. Skip previous rows ($h$):
    * Unit size: $W\times C$ float elements
    * A single row will contain $W$ pixels. Moving to the next row ($h + 1$) means we'll have to jump over all pixels in the current row ($W\times C$) elemnts
    * **Offset cost:** $h\times(W\timesC)
4. Skip previous images ($s$):
    * Unit size: $H\times W\times C$ float elements
    * A full 3D image consists of $H$ rows, each row has $W\times C$ elements. To move onto the next image in the batch ($s+1$), we must skip the entire previous 3D image $H \times W \times C$ elements.
    * **Offset cost:** $s \times (H \times W \times C)$

Hopefully your still with me, for us to find the exact index in flat 1D memory `out_idx`, we add the total jump distances from each dimension: 
$$\text{out\_idx} = \underbrace{s \cdot (H \cdot W \cdot C)}_{\text{Skip } s \text{ images}} + \underbrace{h \cdot (W \cdot C)}_{\text{Skip } h \text{ rows}} + \underbrace{w \cdot C}_{\text{Skip } w \text{ pixels}} + \underbrace{c}_{\text{Step into channel}}$$

## Stage 3: Spatial Window Reduction

This part is actually pretty straight forward, now that our GPU thread knows exactly where its supposed to start at because of `out_idx`. For max pooling, we can calculate exactly where the 0th element is, then traverse across the $fH \times fW$ receptive field. For average pooling, we'll omit this.
```C++
// Not real code below, but an overview of what $reduce_init would look like
// for the three branches
if MAX_OP: 
    int initial_idx = batch_offset + (h_start * W_in + w_start) * C + c;
    float max_val = x[initial_idx];
    int best_idx = initial_idx;

else if MAX_OP_INFERENCE:
    int initial_idx = batch_offset + (h_start * W_in + w_start) * C + c;
    float max_val = x[initial_idx];
    // no best_idx because this branch because we only need max_val to calculate dinputs during training
// We have only this starting value for the loop itself, we don't need to directly calculate the first index, so this is correct
else if AVG_OP:
    float sum_val = 0.0f;
```

If you've noticed, we've actually have our GPU threads map to the output tensor dimensions. If we mapped our calculations to be based on the input tensor dimensions, then we risk race conditions via overlapping windows.Before entering the loop, the thread knows its top-left starting corner in the input tensor based on stride ($sH, sW$):

$$\text{h\_start} = h_{\text{out}} \times sH$$

$$\text{w\_start} = w_{\text{out}} \times sW$$

```C++
    for (int fh = 0; fh < fH; ++fh) {
        int h_in = h_start + fh;
        int row_offset = batch_offset + (h_in * W_pad) * C;
    }
```

* `int h_in = h_start + fH;` `h_in` is the phyiscal row index in the padded input image. 
* `row_offset`: We compute the offset outside the inner loop `fW` because every pixel in the row shares the exact same offset and row height. 

Moving inside the inner for loop
```C++
        for(int fw = 0, fw < fW, ++fW){
            int w_in = w_start + fw;
            int in_idx = row_offset + w_in * C + c;

            $reduce_body
        }
```
* `w_in`: The actual physical column index in the row
* `in_idx`: Converts the 2D window coordinate $(h_{\text{in}}, w_{\text{in}})$ into the flat 1D input memory address:
$$\text{in\_idx} = \text{batch\_offset} + (h_{\text{in}} \times W_{\text{pad}} \times C) + (w_{\text{in}} \times C) + c$$

* `$reduce_body`: This exapnds on the template, which is either finding the max value, finding the max value and saving its index, or create a running total to take the average of the $fH \times fW$ region (from our `OP` dictionaries). 

```C++
            float val = x[in_idx];
            if (val > max_val) { 
                max_val = val; 
                best_idx = in_idx; // Saved for backward pass gradient routing
            }
```

```C++
            sum_val += x[in_idx]:
```

## Stage 4: Global Writeback

After every thread has written to its **own unique slot** in the output tensor defined by `out_idx`, we'll write this back out.
```C++
    //rest of kernel above 
    int out_idx = ((s * H_out + h_out) * W_out + w_out) * C + c;
    $writeback
```
At the start of the kernel, `float* __restrict__ out` was a pointer made to point to the start of the flattened 4D array (1D) in VRAM. `out_idx` is a standard integer representing the offset from this pointer, it doesn't hold any calculated value.
This means, we can update the array at the offset with whatever our desired calculation was inside the for loop.

```C++
    out[out_idx] = max_val
```
In the scenario where we need to store two seperate $(S, H_{out}, W_{out}, C) tensors inside `_MAX_OP`, we'll do the same thing, but update a seperate tensor `max_indices` with this. 
```C++
    out[out_idx] = max_val 
    max_indices[out_idx] = best_idx
```

Finally, in the case of average pooling, we'll do a similar step but update the output value with the average. 

```C++
    out[out_idx] = sum_val / (float)(fH * fW)
```
Only special part here is that we have to cast the integer `fH * fW` into a float to prevent some GPU compiler warnings and improve code clarity. 

